<a href="https://colab.research.google.com/github/reddoma742/Davisson-Germer-DTQEM/blob/main/dtqem_unified_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
dtqem_unified_simulator.py

DTQEM Unified Model Simulator & Model Selection

Generates synthetic matter-wave coherence time data from the Unified Model:
    tau_c = tau_c0 / [ A*m^beta * v^delta * I
                     + B*(1+zeta*N)*dT
                     + C_joint*m^beta * v^delta * I * (1+zeta*N)*dT ]

And tests using Akaike Information Criterion with Correction (AICc)
when the Unified Model is selected over its decoupled sub-models.

The Unified model bridges two historically separate regimes:
    - v18.0-C (environmental: I_path, T)
    - v63.x (particle structure: m, v, N)

Acknowledgment:
    This code was developed with the assistance of AI language models:
    - DeepSeek (critical analysis, methodology validation)
    - Claude (Anthropic) (code writing, derivations, documentation)
    - Arena AI (first-principles derivations of scaling exponents, unification framework)
    Human scientific supervision and validation: Reddouane Berramdane

Version: v1.0 (Unified Simulation Framework, Zenodo-ready)
Date: 2026-06-01

License: Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)
See LICENSE file for full terms.
"""

import numpy as np
import scipy.stats as stats

# Constants
C_LIGHT = 299792458.0
M_U = 1.660539e-27
T_REF = 300.0


class UnifiedDecoherenceModel:
    """
    Implements the Unified DTQEM Equation integrating structural
    scaling and environmental crossover.

    The unified equation combines:
        - Particle properties: mass (m), velocity (v), atom count (N)
        - Environmental conditions: path information (I), temperature (T)
    """
    def __init__(self, tau_c0=9.8e-26, beta=0.44, delta=0.33, zeta=0.005,
                 A=1.0, B=1.5, C_joint=0.8):
        """
        Parameters:
        -----------
        tau_c0 : float
            Base coherence time scale (seconds)
        beta : float
            Mass scaling exponent (0.44 from Debye-Pikovski)
        delta : float
            Velocity scaling exponent (1/3 from van der Waals)
        zeta : float
            Per-atom complexity factor (0.005)
        A : float
            Kinetic term scaling constant
        B : float
            Thermal term scaling constant
        C_joint : float
            Joint-bath crossover coefficient
        """
        self.tau_c0 = tau_c0
        self.beta = beta
        self.delta = delta
        self.zeta = zeta
        self.A = A
        self.B = B
        self.C_joint = C_joint

    def calculate_tau_c(self, m_kg, v_ms, N_atoms, I_path, T_kelvin):
        """
        Calculate coherence time tau_c from particle properties and environment.

        Parameters:
        -----------
        m_kg : float or array_like
            Mass in kilograms
        v_ms : float or array_like
            Velocity in meters per second
        N_atoms : int or array_like
            Number of atoms in the particle
        I_path : float or array_like
            Path information (0 to 1)
        T_kelvin : float or array_like
            Temperature in Kelvin

        Returns:
        --------
        tau_c : float or ndarray
            Coherence time in seconds
        """
        m_scaled = m_kg / M_U
        v_scaled = v_ms / C_LIGHT
        dT = (T_kelvin - T_REF) / T_REF

        # Kinetic term (particle + path information)
        kinetic_term = self.A * (m_scaled ** self.beta) * (v_scaled ** self.delta) * I_path

        # Thermal term (complexity + temperature)
        thermal_term = self.B * (1.0 + self.zeta * N_atoms) * dT

        # Joint term (crossover between particle and environment)
        joint_term = self.C_joint * (m_scaled ** self.beta) * (v_scaled ** self.delta) * I_path * (1.0 + self.zeta * N_atoms) * dT

        denom = kinetic_term + thermal_term + joint_term
        denom = np.maximum(denom, 1e-35)
        return self.tau_c0 / denom


def fit_unified_submodel(m, v, N, I, T, tau_obs, model_type="unified"):
    """
    Fits and calculates RSS for three model variants:

    1. "kinetic_only":
        tau_c = tau_c0 / [A * m^beta * v^delta * I] (K=2)
    2. "decoupled":
        tau_c = tau_c0 / [A*m^beta*v^delta*I + B*(1+zeta*N)*dT] (K=3)
    3. "unified":
        full crossover model (K=4)

    Returns:
        k : int (number of parameters)
        rss : float (residual sum of squares)
    """
    m_scaled = m / M_U
    v_scaled = v / C_LIGHT
    dT = (T - T_REF) / T_REF
    zeta = 0.005
    beta = 0.44
    delta = 0.33

    Y = 1.0 / tau_obs  # Fit the denominator directly (reciprocal of tau_c)

    if model_type == "kinetic_only":
        # Y = A' * m^beta * v^delta * I
        X = (m_scaled ** beta) * (v_scaled ** delta) * I
        slope, _, _, _ = np.linalg.lstsq(X[:, np.newaxis], Y, rcond=None)
        rss = np.sum((Y - X * slope[0])**2)
        k = 2

    elif model_type == "decoupled":
        # Y = A' * m^beta * v^delta * I + B' * (1+zeta*N) * dT
        X1 = (m_scaled ** beta) * (v_scaled ** delta) * I
        X2 = (1.0 + zeta * N) * dT
        X = np.column_stack([X1, X2])
        coeffs, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
        rss = np.sum((Y - X @ coeffs)**2)
        k = 3

    elif model_type == "unified":
        # Y = A'*m^beta*v^delta*I + B'*(1+zeta*N)*dT + C_joint'*m^beta*v^delta*I*(1+zeta*N)*dT
        X1 = (m_scaled ** beta) * (v_scaled ** delta) * I
        X2 = (1.0 + zeta * N) * dT
        X3 = X1 * X2
        X = np.column_stack([X1, X2, X3])
        coeffs, _, _, _ = np.linalg.lstsq(X, Y, rcond=None)
        rss = np.sum((Y - X @ coeffs)**2)
        k = 4

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    return k, rss


def run_unified_selection_simulation(noise_std=0.08, trials=200):
    """
    Simulates the probability of selecting the Unified Model over sub-models
    using AICc as a function of sample size N.

    This reproduces the key statistical finding: larger datasets (N >= 36)
    are required to reliably detect joint-bath coupling.
    """
    np.random.seed(42)
    true_model = UnifiedDecoherenceModel(C_joint=1.2)  # Moderate crossover coupling

    sample_sizes = [10, 20, 30, 40, 60, 100]
    report = {}

    typical_particles = [
        {"m": 720 * M_U, "N": 60},    # C60
        {"m": 840 * M_U, "N": 70},    # C70
        {"m": 1008 * M_U, "N": 84},   # C84
        {"m": 3000 * M_U, "N": 200},  # Giant cluster
        {"m": 6000 * M_U, "N": 450}   # Larger cluster
    ]

    for N in sample_sizes:
        wins = {"kinetic_only": 0, "decoupled": 0, "unified": 0}

        for _ in range(trials):
            # Generate random experimental designs
            m_s, v_s, N_s, I_s, T_s = [], [], [], [], []
            for _ in range(N):
                p = np.random.choice(typical_particles)
                m_s.append(p["m"])
                N_s.append(p["N"])
                v_s.append(np.random.uniform(100.0, 300.0))
                I_s.append(np.random.uniform(0.1, 0.9))
                T_s.append(np.random.uniform(300.0, 450.0))

            m_s, v_s, N_s, I_s, T_s = map(np.array, [m_s, v_s, N_s, I_s, T_s])

            # Generate true coherence times with Gaussian noise
            tau_clean = true_model.calculate_tau_c(m_s, v_s, N_s, I_s, T_s)
            tau_obs = tau_clean + np.random.normal(0, noise_std * tau_clean, N)
            tau_obs = np.maximum(tau_obs, 1e-28)

            # Calculate AICc for the 3 competing models
            aicc_vals = {}
            for m_type in ["kinetic_only", "decoupled", "unified"]:
                k, rss = fit_unified_submodel(m_s, v_s, N_s, I_s, T_s, tau_obs, model_type=m_type)
                s2 = rss / N
                if (N - k - 1) > 0:
                    aicc = N * np.log(s2 if s2 > 0 else 1e-15) + 2*k + (2*k*(k+1)) / (N - k - 1)
                else:
                    aicc = np.inf
                aicc_vals[m_type] = aicc

            # Model with the lowest AICc wins
            best_model = min(aicc_vals, key=aicc_vals.get)
            wins[best_model] += 1

        report[N] = {
            "kinetic_only_%": (wins["kinetic_only"] / trials) * 100,
            "decoupled_%": (wins["decoupled"] / trials) * 100,
            "unified_%": (wins["unified"] / trials) * 100,
            "Reliability": "Low (Prefers Sub-models)" if (wins["unified"]/trials) < 0.50 else (
                "Moderate" if (wins["unified"]/trials) < 0.85 else "High (Unified Model Resolved)"
            )
        }
    return report


if __name__ == "__main__":
    print("=" * 80)
    print("DTQEM Unified Model Selection Simulation via AICc")
    print("=" * 80)

    # Test single prediction
    model = UnifiedDecoherenceModel()
    m_C60 = 720 * M_U
    v_test = 200.0
    N_test = 60
    I_test = 0.5
    T_test = 350.0

    tau = model.calculate_tau_c(m_C60, v_test, N_test, I_test, T_test)
    print(f"\nTest prediction for C60 (m=720u, v=200m/s, N=60, I=0.5, T=350K):")
    print(f"  τ_c = {tau:.4e} seconds\n")

    # Run AICc simulation
    results = run_unified_selection_simulation(noise_std=0.08, trials=200)

    print("Sample Size N | Kinetic Only | Decoupled | Unified | Reliability")
    print("-" * 80)
    for N, stats_data in results.items():
        print(f"     N = {N:<3}    |   {stats_data['kinetic_only_%']:5.1f}%     |   {stats_data['decoupled_%']:5.1f}%   |   {stats_data['unified_%']:5.1f}%   | {stats_data['Reliability']}")

    print("\n" + "=" * 80)
    print("\n📝 Acknowledgment:")
    print("   This code was developed with the assistance of AI models:")
    print("   - DeepSeek (critical analysis, methodology validation)")
    print("   - Claude (Anthropic) (code writing, derivations, documentation)")
    print("   - Arena AI (first-principles derivations, unified framework)")
    print("   Human scientific supervision and validation: Reddouane Berramdane")
    print("\n✅ Unified simulator ready for use.")

DTQEM Unified Model Selection Simulation via AICc

Test prediction for C60 (m=720u, v=200m/s, N=60, I=0.5, T=350K):
  τ_c = 2.3212e-25 seconds

Sample Size N | Kinetic Only | Decoupled | Unified | Reliability
--------------------------------------------------------------------------------
     N = 10     |     0.0%     |    61.5%   |    38.5%   | Low (Prefers Sub-models)
     N = 20     |     0.0%     |    29.5%   |    70.5%   | Moderate
     N = 30     |     0.0%     |    23.0%   |    77.0%   | Moderate
     N = 40     |     0.0%     |    11.0%   |    89.0%   | High (Unified Model Resolved)
     N = 60     |     0.0%     |     4.0%   |    96.0%   | High (Unified Model Resolved)
     N = 100    |     0.0%     |     0.5%   |    99.5%   | High (Unified Model Resolved)


📝 Acknowledgment:
   This code was developed with the assistance of AI models:
   - DeepSeek (critical analysis, methodology validation)
   - Claude (Anthropic) (code writing, derivations, documentation)
   - Arena AI (fi